# 🎬 AI Movie Translate & Dubbing Agent (v2.2) — Official Kaggle GPU Cloud Edition

Kaggle ပေါ်တွင် **NVIDIA Tesla GPUs (2x T4 30GB VRAM / P100 16GB VRAM)** ဖြင့် **Web UI Dashboard** ကို 1-Click အလွယ်တကူ ဖွင့်လှစ်အသုံးပြုနိုင်သော စနစ် ဖြစ်ပါသည်။

### ⚙️ Kaggle တွင် အသုံးမပြုမီ အရေးကြီးသော ကြိုတင်ပြင်ဆင်ချက် (2 Steps):
1. **GPU ဖွင့်ပါ:** ညာဘက် Sidebar ရှိ `Notebook Settings` > `Accelerator` > **GPU T4 x2** (သို့မဟုတ် **GPU P100**) ကို ရွေးချယ်ပါ။
2. **Internet ဖွင့်ပါ:** ညာဘက် Sidebar ရှိ `Notebook Settings` > `Internet` > **Internet on** ကို အမှန်ခြစ်ဖွင့်ပေးပါ။

### 🌟 Kaggle Cloud GPU v2.2 စနစ်သစ် အဓိက အားသာချက်များ:
* 🎯 **100% Frame-Accurate Audio Sync & Zero Cumulative Drift:** မူရင်း စက္ကန့်အတိုင်း အတိအကျကိုက်ညီသော 44.1kHz Multi-Track Audio Positioning Engine (အချိန်ကွာဟမှု 0.000s အထိ တိကျခြင်း)။
* 🎙️ **Natural Human Voice Sweet Spot (`+18%`) & Zero Robotic Sound:** စက်ရုပ်သံ/သံပြာသံ လုံးဝမထွက်စေဘဲ သဘာဝကျပြီး ဆွဲဆောင်မှုရှိသော Movie Recap အမြန်နှုန်း။
* 🔇 **100% Muted Original Audio on `--skip-demucs` + Looped BGM:** မူရင်းအင်္ဂလိပ်အသံ (၀%) လုံးဝမထွက်စေဘဲ Cinematic Tension BGM ဖြင့် ပေါင်းစပ်ပေးခြင်း။
* 🎮 **Dual T4 GPUs (30GB VRAM):** Whisper CUDA FP16, Demucs Vocal Separation နှင့် FFmpeg NVENC ဖြင့် 10x ပိုမိုမြန်ဆန်ခြင်း။
* ⏳ **30 Hours Free GPU / Week:** တစ်ပတ်လျှင် နာရီ ၃၀ အထိ Cloud GPU အခမဲ့ သုံးစွဲနိုင်ခြင်း။
* ⏱️ **12 Hours Long Session:** တစ်ကြိမ် Run လျှင် ၁၂ နာရီ ဆက်တိုက် Batch Processing ဖြင့် ဇာတ်ကားပေါင်းများစွာ ထုတ်လုပ်နိုင်ခြင်း။
* 🍪 **Multi-Platform Auto Downloader:** YouTube, DramaBox (`dramaboxdb.com`), ReelShort (`reelshort.com`) link များ တိုက်ရိုက်ဒေါင်းလုဒ်ဆွဲနိုင်ခြင်း။
* 📱 **Facebook Reels & TikTok (9:16 Full HD Canvas):** Brand Header (`🎬 Pai AI Movie Studio`)၊ Golden Hook Title နှင့် Safe-zone Subtitles (`box_black`)။
* 🎨 **Subtitle Style Presets:** Cinema Box (Netflix), TikTok/Reels Yellow, Classic White, Cyber Cyan, Thriller Crimson ပုံစံ ၅ မျိုး Web UI မှ အလွယ်တကူ ရွေးချယ်နိုင်ခြင်း။


In [ ]:
# @title 🚀 1. Launch Web UI Dashboard (Kaggle Dedicated GPU Mode)
# @markdown Run this cell to start the Web UI Dashboard powered by Kaggle's Dedicated GPUs (T4 x2 or P100):
# @markdown *(Optional) မိမိ၏ Gemini API Keys (ကော်မာ သို့မဟုတ် မျဉ်းအသစ် ခံနိုင်သည် - မထည့်ပါက Kaggle Secrets မှ အလိုအလျောက် ရယူပါမည်):*
GEMINI_API_KEYS = "" # @param {type:"string"}
# @markdown *(Optional) YouTube/DramaBox cookies.txt စာသားများကို ဤနေရာတွင် တိုက်ရိုက် Paste ထည့်နိုင်ပါသည်:*
COOKIES_TXT = "" # @param {type:"string"}

import os
import sys
import json
import subprocess
import time
import re
import shutil
import socket
import glob
import torch
from IPython.display import display, HTML

# 0. Strict Kaggle Environment & Dedicated GPU Verification
if not torch.cuda.is_available():
    print('\n❌ [CRITICAL NOTICE] Kaggle တွင် GPU မရွေးချယ်ထားပါ!')
    print('👉 ကျေးဇူးပြု၍ ညာဘက် Sidebar ရှိ: Notebook Settings > Accelerator > "GPU T4 x2" (သို့မဟုတ် "GPU P100") ကို ရွေးချယ်ပြီးမှ ပြန်လည် Run ပေးပါ ခင်ဗျာ。\n')
    raise RuntimeError("Kaggle requires GPU Accelerator. Go to Notebook Settings > Accelerator > GPU T4 x2 or GPU P100.")

gpu_count = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(gpu_count)) / (1024**3)
print(f'🚀 [Kaggle Dedicated GPU Active] {gpu_count}x GPU Detected: {", ".join(gpu_names)} ({total_vram_gb:.1f} GB Total VRAM) — 100% GPU Acceleration Ready!')

# Check Kaggle Internet Toggle
try:
    with socket.create_connection(("8.8.8.8", 53), timeout=3):
        print("🌐 [Internet Connection] Verified OK!")
except OSError:
    print('\n❌ [CRITICAL NOTICE] Kaggle Internet Toggle ပိတ်ထားပါသည်!')
    print('👉 ကျေးဇူးပြု၍ ညာဘက် Sidebar ရှိ: Notebook Settings > Internet > "Internet on" ကို အမှန်ခြစ်ဖွင့်ပေးပါ ခင်ဗျာ。\n')
    raise RuntimeError("Kaggle requires Internet access. Enable: Notebook Settings > Internet > Internet on.")

# Set Dedicated GPU environment flags (Multi-GPU friendly)
os.environ["FORCE_GPU"] = "true"
if gpu_count > 1:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", ",".join(str(i) for i in range(gpu_count)))
else:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

project_dir = "/kaggle/working/ai-translate-agent"

# 1. Setup Repository in /kaggle/working
if not os.path.exists(project_dir):
    print("[*] 1/4 Cloning repository...")
    subprocess.run(["git", "clone", "https://github.com/paipai1999/ai-translate-agent.git", project_dir], check=True)
else:
    print("[*] 1/4 Updating repository to latest commit...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=project_dir, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=project_dir, check=True)

os.chdir(project_dir)

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Create config.json from config.example.json if missing
cfg_path = os.path.join(project_dir, "config.json")
cfg_example = os.path.join(project_dir, "config.example.json")
if not os.path.exists(cfg_path) and os.path.exists(cfg_example):
    shutil.copy(cfg_example, cfg_path)

# Auto-detect Gemini API Key from Kaggle Secrets if form field is empty
if not (GEMINI_API_KEYS and str(GEMINI_API_KEYS).strip()):
    try:
        from kaggle_secrets import UserSecretsClient
        _usc = UserSecretsClient()
        for _sk in ["GEMINI_API_KEYS", "GEMINI_API_KEY", "GEMINI_KEY"]:
            try:
                _sec = _usc.get_secret(_sk)
                if _sec and str(_sec).strip():
                    GEMINI_API_KEYS = str(_sec).strip()
                    print(f"🔑 [Kaggle Secrets] Auto-detected '{_sk}' from Kaggle Secrets!")
                    break
            except Exception:
                pass
    except Exception:
        pass

# Update GEMINI_API_KEYS in config.json and os.environ
if GEMINI_API_KEYS and str(GEMINI_API_KEYS).strip():
    try:
        with open(cfg_path, "r", encoding="utf-8") as _cf:
            _cdata = json.load(_cf)
        _parsed = [k.strip() for k in str(GEMINI_API_KEYS).replace("\n", ",").split(",") if k.strip()]
        if _parsed:
            _cdata.setdefault("gemini", {})["api_keys"] = _parsed
            with open(cfg_path, "w", encoding="utf-8") as _cf:
                json.dump(_cdata, _cf, indent=4)
            os.environ["GEMINI_API_KEYS"] = ",".join(_parsed)
            os.environ["GEMINI_API_KEY"] = _parsed[0]
            print(f"🔑 [API Keys] Configured {len(_parsed)} Gemini API key(s) successfully!")
    except Exception as _ke:
        print(f"[WARN] Could not set GEMINI_API_KEYS: {_ke}")

# Auto-configure Cookies from Form, Kaggle Datasets, or /kaggle/working
os.makedirs('assets', exist_ok=True)
if COOKIES_TXT and str(COOKIES_TXT).strip():
    with open("cookies.txt", "w", encoding="utf-8") as _ck:
        _ck.write(str(COOKIES_TXT).strip())
    with open(os.path.join("assets", "cookies.txt"), "w", encoding="utf-8") as _ck:
        _ck.write(str(COOKIES_TXT).strip())
    with open("/kaggle/working/cookies.txt", "w", encoding="utf-8") as _ck:
        _ck.write(str(COOKIES_TXT).strip())
    print("🍪 [Cookies] Installed cookies.txt from notebook form input!")
else:
    restored_cookie = False
    for k_c in glob.glob('/kaggle/input/**/cookies*.txt', recursive=True):
        if os.path.exists(k_c) and os.path.getsize(k_c) > 10:
            shutil.copy2(k_c, "cookies.txt")
            shutil.copy2(k_c, os.path.join("assets", "cookies.txt"))
            shutil.copy2(k_c, "/kaggle/working/cookies.txt")
            print(f"🍪 [Cookies Auto-Restore] Found & restored cookies from Kaggle Dataset: {k_c}")
            restored_cookie = True
            break
    if not restored_cookie and os.path.exists("/kaggle/working/cookies.txt"):
        shutil.copy2("/kaggle/working/cookies.txt", "cookies.txt")
        shutil.copy2("/kaggle/working/cookies.txt", os.path.join("assets", "cookies.txt"))
        print("🍪 [Cookies Auto-Restore] Restored cookies from /kaggle/working/cookies.txt")

os.makedirs("temp", exist_ok=True)
os.makedirs("movies", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Symlink /kaggle/working/outputs -> ai-translate-agent/outputs for instant Kaggle UI output download
if not os.path.exists("/kaggle/working/outputs"):
    try:
        os.symlink(os.path.abspath("outputs"), "/kaggle/working/outputs")
        print("🔗 [Outputs] Linked /kaggle/working/outputs -> ai-translate-agent/outputs for instant Kaggle UI download!")
    except Exception:
        pass

# 2. Install System Dependencies, Myanmar Fonts & xz-utils
print("[*] 2/4 Checking & Installing system dependencies & Myanmar fonts...")
subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg fonts-sil-padauk fonts-noto-cjk fonts-noto-core xz-utils tar curl nodejs > /dev/null 2>&1", shell=True)
subprocess.run("fc-cache -f > /dev/null 2>&1", shell=True)

# Install GPU-accelerated NVENC FFmpeg for NVIDIA Dedicated GPU
def _setup_nvenc_ffmpeg():
    # Configure LD_LIBRARY_PATH for NVIDIA CUDA & NVENC driver libraries on Linux
    ld_paths = ["/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/lib64", "/usr/local/nvidia/lib64", "/usr/local/cuda/targets/x86_64-linux/lib"]
    cur_ld = os.environ.get("LD_LIBRARY_PATH", "")
    extra_ld = [p for p in ld_paths if os.path.exists(p) and p not in cur_ld]
    if extra_ld:
        os.environ["LD_LIBRARY_PATH"] = ":".join(extra_ld) + ((":" + cur_ld) if cur_ld else "")
    if os.path.exists("/usr/local/bin/ffmpeg") and os.path.getsize("/usr/local/bin/ffmpeg") > 10000000:
        try:
            chk = subprocess.run(["/usr/local/bin/ffmpeg", "-y", "-f", "lavfi", "-i", "nullsrc=s=64x64:d=0.1", "-c:v", "h264_nvenc", "-f", "null", "-"], capture_output=True, timeout=4)
            if chk.returncode == 0:
                print("🚀 [OK] NVIDIA NVENC FFmpeg already active and verified!")
                os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")
                os.environ["IMAGEIO_FFMPEG_EXE"] = "/usr/local/bin/ffmpeg"
                return
        except Exception:
            pass

    print("[*] Installing NVIDIA NVENC GPU-accelerated FFmpeg...")
    urls = [
        "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz",
        "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-n7.1-latest-linux64-gpl.tar.xz",
    ]
    try:
        os.makedirs("/tmp/ff_build", exist_ok=True)
        downloaded = False
        for u in urls:
            try:
                res = subprocess.run(f"curl -L -f -s -A 'Mozilla/5.0' {u} -o /tmp/ff_build/ffmpeg.tar.xz", shell=True, timeout=90)
                if res.returncode == 0 and os.path.exists("/tmp/ff_build/ffmpeg.tar.xz") and os.path.getsize("/tmp/ff_build/ffmpeg.tar.xz") > 10000000:
                    downloaded = True
                    break
            except Exception:
                continue
        if not downloaded:
            print("[WARN] Could not download static NVENC FFmpeg build.")
            return
        subprocess.run("tar -xf /tmp/ff_build/ffmpeg.tar.xz -C /tmp/ff_build", shell=True, check=True, timeout=45)
        subprocess.run("find /tmp/ff_build -type f -name ffmpeg -exec cp -f {} /usr/local/bin/ffmpeg \\;", shell=True, check=True)
        subprocess.run("find /tmp/ff_build -type f -name ffprobe -exec cp -f {} /usr/local/bin/ffprobe \\;", shell=True, check=True)
        subprocess.run("chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe", shell=True, check=True)
        subprocess.run("rm -rf /tmp/ff_build", shell=True)
        os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")
        os.environ["IMAGEIO_FFMPEG_EXE"] = "/usr/local/bin/ffmpeg"
        
        test_nvenc = subprocess.run(["/usr/local/bin/ffmpeg", "-y", "-f", "lavfi", "-i", "nullsrc=s=64x64:d=0.1", "-c:v", "h264_nvenc", "-f", "null", "-"], capture_output=True, timeout=4)
        if test_nvenc.returncode == 0:
            print("🚀 [OK] NVIDIA NVENC GPU Encoder successfully verified and active!")
        else:
            print("ℹ️ [Notice] NVENC FFmpeg installed; driver ready.")
    except Exception as e:
        print(f"[WARN] NVENC FFmpeg setup notice: {e}")

_setup_nvenc_ffmpeg()

# Ensure yt-dlp and requirements are installed
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Install Cloudflared Tunnel
if not os.path.exists("/usr/local/bin/cloudflared") or os.path.getsize("/usr/local/bin/cloudflared") < 10000:
    subprocess.run("curl -L -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)

# Stop any existing server processes
subprocess.run("pkill -f 'web_ui.py' || true", shell=True)
subprocess.run("pkill -f 'cloudflared' || true", shell=True)

# 3. Start Web UI Server (Host 0.0.0.0 Bind)
print("[*] 3/4 Starting Web UI Server...")
server_proc = subprocess.Popen(
    [sys.executable, "web_ui.py", "--host", "0.0.0.0", "--port", "5000"],
    cwd=project_dir,
    stdout=open("/kaggle/working/web_ui.log", "a", buffering=1),
    stderr=subprocess.STDOUT
)

# Wait until Web UI port 5000 is actively accepting connections
server_ready = False
for _ in range(35):
    if server_proc.poll() is not None:
        break
    try:
        with socket.create_connection(("127.0.0.1", 5000), timeout=1):
            server_ready = True
            break
    except OSError:
        time.sleep(1)

if not server_ready:
    print("❌ Web UI failed to start! Crash log details:")
    if os.path.exists("/kaggle/working/web_ui.log"):
        with open("/kaggle/working/web_ui.log", "r") as f:
            print(f.read())
else:
    print("[*] 4/4 Connecting Cloudflare Secure Tunnel...")
    tunnel_log_path = "/kaggle/working/cloudflared.log"
    tunnel_log_file = open(tunnel_log_path, "a", buffering=1)
    tunnel_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--protocol", "http2", "--url", "http://127.0.0.1:5000"],
        stdout=tunnel_log_file,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    tunnel_url = None
    start_t = time.time()
    while time.time() - start_t < 45:
        if os.path.exists(tunnel_log_path):
            with open(tunnel_log_path, "r", errors="ignore") as _f:
                _content = _f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', _content)
            if match:
                tunnel_url = match.group(0)
                break
        time.sleep(0.5)

    # Allow 3 seconds for DNS propagation
    time.sleep(3)

    if tunnel_url:
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #0d1117, #161b22); border: 2px solid #20beff; border-radius: 14px; padding: 26px; text-align: center; margin: 20px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 8px 24px rgba(0,0,0,0.5);">
            <div style="font-size: 38px; margin-bottom: 6px;">🎬</div>
            <h2 style="color: #20beff; margin: 0 0 10px 0; font-size: 22px;">Kaggle GPU Web UI Dashboard အဆင်သင့် ဖြစ်ပါပြီ!</h2>
            <p style="color: #c9d1d9; font-size: 14px; margin: 0 0 18px 0;">အောက်ပါ Cloudflare Tunnel ခလုတ်ကို နှိပ်၍ Web UI ကို ဖွင့်ပါ 👇</p>
            <div style="display: flex; justify-content: center; flex-wrap: wrap; gap: 10px;">
                <a href="{tunnel_url}" target="_blank" style="background: linear-gradient(135deg, #20beff, #005bb5); color: #ffffff; font-weight: bold; font-size: 18px; padding: 14px 32px; border-radius: 10px; text-decoration: none; display: inline-block; box-shadow: 0 4px 16px rgba(32, 190, 255, 0.4); margin: 6px;">
                    🚀 Open Web UI Dashboard ↗️
                </a>
            </div>
            <div style="margin-top: 16px; font-size: 13px; color: #8b949e;">
                💡 <i>Kaggle တွင် အပတ်စဉ် <b>30 Hours Free GPU</b> ရရှိပြီး <b>12 Hours ဆက်တိုက်</b> အသုံးပြုနိုင်ပါသည်။</i>
            </div>
        </div>
        """))
        print(f"\n👉 Public Web UI URL: {tunnel_url}")
        print("\n🟢 Web UI Server is LIVE! Click the link above to open Dashboard.")
        print("💡 Keep this cell running while using the Web UI. (To stop, click ⏹️ Interrupt)")
        try:
            while True:
                time.sleep(2)
                if server_proc.poll() is not None:
                    print("\n❌ Web UI server stopped!")
                    break
                if tunnel_proc.poll() is not None:
                    print("\n❌ Tunnel disconnected!")
                    break
        except KeyboardInterrupt:
            print("\n🛑 Stopping Web UI and Tunnel...")
            server_proc.terminate()
            tunnel_proc.terminate()
    else:
        print("❌ Cloudflare Tunnel failed to obtain a public URL within 45 seconds!")
        if os.path.exists(tunnel_log_path):
            with open(tunnel_log_path, "r", errors="ignore") as _f:
                print(f"Tunnel Log details:\n{_f.read()}")


In [ ]:
# @title 💻 2. Command Line (CLI) Direct Run (Alternative)
# @markdown Web UI မသုံးဘဲ Notebook ထဲမှ တိုက်ရိုက် Recap ဗီဒီယို ထုတ်ယူလိုပါက အောက်ပါအတိုင်း ဖြည့်သွင်း၍ Run နိုင်ပါသည်:
video_input = "https://youtu.be/KmYSM5knNV8" #@param {type:"string"}
video_format = "9:16" #@param ["9:16", "16:9", "both"]
subtitle_style = "box_black" #@param ["box_black", "yellow_pop", "white_stroke", "cyan_cyber", "crimson_box"]
voice = "my-MM-ThihaNeural" #@param ["my-MM-ThihaNeural", "my-MM-NilarNeural"]
resolution = "1080p" #@param ["1080p", "720p"]
skip_demucs = True #@param {type:"boolean"}
subtitle_mode = "burn" #@param ["burn", "none"]
thumbnail_title = "" #@param {type:"string"}
watermark_text = "Pai Movie AI Studio" #@param {type:"string"}
source_language = "auto" #@param ["auto", "zh", "en", "th", "ko", "ja"]
resume = True #@param {type:"boolean"}
thumbnail_intro = False #@param {type:"boolean"}

import subprocess, sys, os

project_dir = "/kaggle/working/ai-translate-agent"
if not os.path.exists(project_dir):
    print("⚠️ Repository not initialized yet! Please run Cell 1 first to install dependencies.")
else:
    os.chdir(project_dir)
    if not os.environ.get("GEMINI_API_KEYS"):
        try:
            from kaggle_secrets import UserSecretsClient
            _usc = UserSecretsClient()
            for _sk in ["GEMINI_API_KEYS", "GEMINI_API_KEY", "GEMINI_KEY"]:
                try:
                    _sec = _usc.get_secret(_sk)
                    if _sec and str(_sec).strip():
                        os.environ["GEMINI_API_KEYS"] = str(_sec).strip()
                        os.environ["GEMINI_API_KEY"] = str(_sec).strip().split(",")[0].strip()
                        break
                except Exception:
                    pass
        except Exception:
            pass

    cmd = [
        sys.executable, "main.py",
        "--input", video_input,
        "--format", video_format,
        "--sub-style", subtitle_style,
        "--voice", voice,
        "--res", resolution,
        "--sub-mode", subtitle_mode,
        "--source-lang", source_language
    ]
    if not resume:
        cmd.append("--fresh")
    if thumbnail_intro:
        cmd.append("--thumbnail-intro")
    else:
        cmd.append("--no-thumbnail-intro")
    if skip_demucs:
        cmd.append("--skip-demucs")
    if thumbnail_title.strip():
        cmd.extend(["--thumb-title", thumbnail_title.strip()])
    if watermark_text.strip():
        cmd.extend(["--watermark-text", watermark_text.strip()])
    else:
        cmd.append("--no-watermark")

    print(f"🚀 Running: {' '.join(cmd)}")
    result = subprocess.run(cmd)
    if result.returncode == 0:
        print("Pipeline complete! Run Cell 3 to download.")
    else:
        print("Pipeline exited with code", result.returncode)


In [ ]:
# @title 📁 3. View & 1-Click Download Generated Outputs
# @markdown Run this cell to view and download all generated videos, scripts, subtitles, and thumbnails:
import os, glob
from IPython.display import display, HTML, FileLink

search_patterns = [
    "/kaggle/working/ai-translate-agent/outputs/**/*.*",
    "/kaggle/working/outputs/**/*.*"
]
all_files = []
for pat in search_patterns:
    all_files.extend(glob.glob(pat, recursive=True))

output_files = sorted(list(set([
    f for f in all_files 
    if os.path.isfile(f) and f.lower().endswith(('.mp4', '.srt', '.ass', '.txt', '.png', '.jpg', '.json', '.vtt'))
])))


# Priority sort: final_reels > final_recap > thumbnail > script > seo
PRIORITY_SUFFIXES = ('final_reels.mp4', 'final_recap.mp4', 'thumbnail.jpg', 'final_recap_script.txt', 'seo_metadata.json')
priority_files = [f for f in output_files if any(f.endswith(s) for s in PRIORITY_SUFFIXES)]
other_files = [f for f in output_files if f not in priority_files and not f.endswith('pipeline.log')]
output_files = priority_files + other_files

if not output_files:
    print("ℹ️ No output files found yet. Run the Web UI (Cell 1) or CLI (Cell 2) to generate recaps!")
    print("💡 Output files will be stored in '/kaggle/working/outputs' and displayed in Kaggle's Data > Output sidebar.")
else:
    print(f"🎉 Found {len(output_files)} generated file(s):\n")
    html_links = "<div style='background:#161b22; padding:18px; border-radius:10px; border:1px solid #30363d; font-family:-apple-system, sans-serif;'>"
    html_links += "<h3 style='color:#58a6ff; margin-top:0;'>📥 Click to Download Generated Files:</h3>"
    for f in output_files:
        size_mb = os.path.getsize(f) / (1024*1024) if os.path.exists(f) else 0
        fname = os.path.basename(f)
        rel_path = os.path.relpath(f, "/kaggle/working")
        icon = "🎬" if fname.endswith(".mp4") else ("📝" if fname.endswith((".srt", ".ass", ".txt")) else "🖼️")
        print(f"{icon} {fname} ({size_mb:.2f} MB) -> {f}")
        html_links += f'<div style="margin:8px 0;"><a href="/files/{rel_path}" download="{fname}" style="color:#38d9a9; font-weight:bold; text-decoration:none; font-size:15px;">⬇️ {icon} Download {fname} ({size_mb:.2f} MB)</a></div>'
    html_links += "<div style='margin-top:14px; font-size:13px; color:#8b949e;'>💡 <i>Kaggle ၏ ညာဘက် 'Data > Output' sidebar တွင်လည်း ဖိုင်အားလုံးကို တိုက်ရိုက် ကြည့်ရှု/Download ပြုလုပ်နိုင်ပါသည်။</i></div>"
    html_links += "</div>"
    display(HTML(html_links))
